<a href="https://colab.research.google.com/github/MhThorq/AnomaliEwallet/blob/main/Fraud_Detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload() # Unggah file kaggle.json di sini

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset langsung ke Colab
!kaggle competitions download -c ieee-fraud-detection
!unzip ieee-fraud-detection.zip

Saving kaggle.json to kaggle.json
  0% 0.00/118M [00:00<?, ?B/s]
100% 118M/118M [00:00<00:00, 1.47GB/s]
Archive:  ieee-fraud-detection.zip
  inflating: sample_submission.csv   
  inflating: test_identity.csv       
  inflating: test_transaction.csv    
  inflating: train_identity.csv      
  inflating: train_transaction.csv   


In [2]:
import pandas as pd
import numpy as np
import gc # Garbage Collector untuk membersihkan RAM

def reduce_mem_usage(df):
    """ Fungsi untuk mengecilkan ukuran memori dataframe """
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memori berkurang menjadi {end_mem:.2f} MB ({((start_mem - end_mem) / start_mem * 100):.1f}% berkurang)')
    return df

In [3]:
# Load data
train_transaction = pd.read_csv('train_transaction.csv')
train_identity = pd.read_csv('train_identity.csv')

# Optimasi memori segera setelah load
train_transaction = reduce_mem_usage(train_transaction)
train_identity = reduce_mem_usage(train_identity)

# Gabungkan data (Merging)
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

# Hapus variabel lama untuk mengosongkan RAM
del train_transaction, train_identity
gc.collect()

print(f"Data gabungan siap dengan bentuk: {train.shape}")

/tmp/ipython-input-398462740.py:21: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipython-input-398462740.py:21: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipython-input-398462740.py:21: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipython-input-398462740.py:21: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipython-input-398462740.py:21: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipython-input-398462740.py:21: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipython-input-398462740.py:21: RuntimeWarning: overfl

Memori berkurang menjadi 542.35 MB (69.4% berkurang)
Memori berkurang menjadi 25.86 MB (42.7% berkurang)


/tmp/ipython-input-398462740.py:21: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:


Data gabungan siap dengan bentuk: (590540, 434)


In [4]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

# Gunakan opsi ini agar semua kolom terlihat saat di-display
pd.set_option('display.max_columns', 500)

In [11]:
import os

output_dir = '/content/drive/My Drive/Fraud-Dataset'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

train.to_pickle(os.path.join(output_dir, 'processed_data.pkl'))